# 📊 Exploratory Data Analysis — Telco Customer Churn

This notebook performs a comprehensive EDA on the IBM Telco Customer Churn dataset.
We explore distributions, correlations, churn patterns, and service-level insights
to build intuition before modeling.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.figsize'] = (12, 6)
print('Libraries loaded ✅')

## 1. Load & Inspect Data

In [ ]:
df = pd.read_csv('../data/raw/telco_churn.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include='all').T

In [ ]:
# Check missing values
missing = df.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0] if missing.sum() > 0 else 'No missing values ✅')

# TotalCharges has blanks stored as spaces
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print(f'\nTotalCharges NaN after conversion: {df["TotalCharges"].isna().sum()}')
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)

## 2. Target Variable Distribution

In [ ]:
churn_counts = df['Churn'].value_counts()
print(churn_counts)
print(f'\nChurn Rate: {churn_counts["Yes"] / len(df) * 100:.1f}%')

fig = px.pie(values=churn_counts.values, names=churn_counts.index,
             title='Churn Distribution', hole=0.5,
             color_discrete_sequence=['#27ae60', '#e05c5c'])
fig.show()

## 3. Numerical Feature Distributions

In [ ]:
num_features = ['tenure', 'MonthlyCharges', 'TotalCharges']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, feat in zip(axes, num_features):
    for label, color in [('No', '#27ae60'), ('Yes', '#e05c5c')]:
        subset = df[df['Churn'] == label][feat]
        ax.hist(subset, bins=30, alpha=0.6, label=label, color=color)
    ax.set_title(f'{feat} by Churn Status', fontweight='bold')
    ax.set_xlabel(feat)
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Box plots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, feat in zip(axes, num_features):
    sns.boxplot(data=df, x='Churn', y=feat, ax=ax,
                palette={'No': '#27ae60', 'Yes': '#e05c5c'})
    ax.set_title(f'{feat} by Churn', fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Categorical Feature Analysis

In [ ]:
cat_features = ['gender', 'Partner', 'Dependents', 'PhoneService',
                'InternetService', 'Contract', 'PaperlessBilling', 'PaymentMethod']

fig, axes = plt.subplots(2, 4, figsize=(22, 10))
for ax, feat in zip(axes.flat, cat_features):
    ct = pd.crosstab(df[feat], df['Churn'], normalize='index') * 100
    ct.plot(kind='bar', stacked=True, ax=ax,
            color=['#27ae60', '#e05c5c'], edgecolor='white')
    ax.set_title(feat, fontweight='bold')
    ax.set_ylabel('Percentage')
    ax.legend(title='Churn')
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## 5. Churn by Contract Type & Tenure

In [ ]:
# Churn by Contract
contract_churn = df.groupby('Contract')['Churn'].apply(
    lambda x: (x == 'Yes').mean() * 100
).reset_index()
contract_churn.columns = ['Contract', 'Churn Rate (%)']

fig = px.bar(contract_churn, x='Contract', y='Churn Rate (%)',
             color='Contract', text_auto='.1f',
             title='Churn Rate by Contract Type',
             color_discrete_sequence=['#1f4e79', '#2e86ab', '#e05c5c'])
fig.show()

In [ ]:
# Tenure groups
bins = [0, 12, 24, 48, 60, float('inf')]
labels = ['New (0-12)', 'Growing (13-24)', 'Mature (25-48)', 'Loyal (49-60)', 'Champion (61+)']
df['tenure_group'] = pd.cut(df['tenure'], bins=bins, labels=labels, right=True)

tenure_churn = df.groupby('tenure_group')['Churn'].apply(
    lambda x: (x == 'Yes').mean() * 100
).reset_index()
tenure_churn.columns = ['Tenure Group', 'Churn Rate (%)']

fig = px.bar(tenure_churn, x='Tenure Group', y='Churn Rate (%)',
             color='Churn Rate (%)', text_auto='.1f',
             title='Churn Rate by Tenure Group',
             color_continuous_scale=['#27ae60', '#f39c12', '#e05c5c'])
fig.show()

## 6. Service Subscription Analysis

In [ ]:
services = ['OnlineSecurity', 'TechSupport', 'OnlineBackup',
            'DeviceProtection', 'StreamingTV', 'StreamingMovies']

svc_data = []
for svc in services:
    for val in df[svc].unique():
        subset = df[df[svc] == val]
        rate = (subset['Churn'] == 'Yes').mean() * 100
        svc_data.append({'Service': svc, 'Status': val, 'Churn Rate (%)': rate})

svc_df = pd.DataFrame(svc_data)

fig = px.bar(svc_df, x='Service', y='Churn Rate (%)', color='Status',
             barmode='group', text_auto='.1f',
             title='Churn Rate by Service Subscription',
             color_discrete_sequence=['#1f4e79', '#e05c5c', '#2e86ab'])
fig.show()

## 7. Correlation Heatmap

In [ ]:
df['Churn_num'] = (df['Churn'] == 'Yes').astype(int)
corr_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'Churn_num']
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, linewidths=1, ax=ax)
ax.set_title('Correlation Heatmap', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Key Insights Summary

| # | Insight |
|---|--------|
| 1 | **26.5%** of customers have churned — a significant business problem. |
| 2 | **Month-to-month** contracts have ~42% churn vs ~3% for two-year contracts. |
| 3 | **New customers** (0-12 months tenure) churn at the highest rate (~47%). |
| 4 | **Fiber optic** internet users churn more than DSL users. |
| 5 | **Electronic check** payers churn at ~45% vs ~15-18% for auto-pay methods. |
| 6 | Customers **without TechSupport or OnlineSecurity** churn 2× more. |
| 7 | **MonthlyCharges** is positively correlated with churn; **tenure** is negatively correlated. |

In [ ]:
print('\n🎉  Exploratory Data Analysis complete!')